# 4. Ensembles de Arboles de Decision

Un arbol de decisión es un modelo débil, el aumento del poder predictivo proviene al ensamblar varios arboles de decisión.
<br> Si promedio n arboles identicos, el resultados es exactamente el mismo que utilizar un solo arbol, necesito PERTURBAR cada arbol para disponer de variablidad

la variabilidad provendrá de estas fuentes:


*   Perturbar el dataset
*   Perturbar el algoritmo del arbol
*   Perturbar el dataset y el algoritmo del arbol al mismo tiempo

Se verán estos tres algoritmos


*   Arboles Azarosos
*   Random Forest
*   Gradient Boosting of Decision Trees

#### 4.01 Seteo del ambiente local
Configuración de directorios de trabajo y descarga del dataset directamente al entorno local (sin requerir Google Drive ni credenciales de Kaggle).

In [1]:
# Determinar la raíz del proyecto de forma absoluta
find_project_root <- function() {
  curr <- normalizePath(getwd(), winslash = "/")
  while (curr != "/" && curr != dirname(curr)) {
    if (file.exists(file.path(curr, "Dockerfile")) || dir.exists(file.path(curr, "src"))) {
      return(curr)
    }
    curr <- dirname(curr)
  }
  return(normalizePath(getwd(), winslash = "/"))
}

dir_base <- if (dir.exists("/workspace")) "/workspace" else find_project_root()
setwd(dir_base)

# Crear carpetas locales para datasets y experimentos
dir.create(file.path(dir_base, "datasets"), showWarnings = FALSE, recursive = TRUE)
dir.create(file.path(dir_base, "exp"), showWarnings = FALSE, recursive = TRUE)

# Descargar el dataset si no existe localmente
url_dataset <- "https://storage.googleapis.com/open-courses/utn2026-b40a/dataset_pequeno.csv"
archivo_dataset <- file.path(dir_base, "datasets", "dataset_pequeno.csv")

if (!file.exists(archivo_dataset)) {
  cat("Descargando dataset_pequeno.csv...\n")
  download.file(url_dataset, destfile = archivo_dataset, mode = "wb")
  cat("Descarga completada exitosamente.\n")
} else {
  cat("El dataset ya se encuentra disponible en:", archivo_dataset, "\n")
}

El dataset ya se encuentra disponible en: /workspace/datasets/dataset_pequeno.csv 




---



## 4.02 Arboles Azarosos

Arboles Azarosos es el nombre de un algoritmo trivial (por favor NO confundir con Random Forest)
Qué tipo de perturbaciones se realizan en Arboles Azarosos
* Se perturba el dataset
* No se perturba el algoritmo, es siempre rpart original

Cada  arbolito de  Arboles Azarosos se entrena sobre un dataset perturbado,  que tiene exactamente la misma cantidad de registros pero solo un subconjunto de los atributos (campos)  del dataset, tomados al azar, de los originales.
<br> En esta primera corrida, se construira cada arbol en un dataset utilizando el 50% de los campos

limpio el ambiente de R

In [2]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Mon Aug 24 15:32:02 2026"

In [3]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,661702,35.4,1454514,77.7,1454514,77.7
Vcells,1232192,9.5,8388608,64.0,1975054,15.1


In [4]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

Loading required package: data.table

Loading required package: rpart



Aqui debe cargar SU semilla primigenia

In [5]:
PARAM <- list()
PARAM$semilla_primigenia <- 115879  # Reemplazar por su semilla primigenia

# Cantidad de arbolitos por ensemble (fijado en 32 para la Tarea 03)
PARAM$num_trees_max <- 32

# Espacio de búsqueda / Grid de hiperparámetros a iterar
# minbucket_fraction define minbucket como fracción de minsplit (ej: 0.1, 0.2, 0.3, 0.4)

# Valores del experimento 420_00
# PARAM$grid <- list(
#   feature_fraction = c(0.2, 0.3, 0.5, 0.7),
#   maxdepth = c(6, 8, 10, 12),
#   minsplit = c(100, 200, 400, 600, 800),
#   minbucket_fraction = c(0.2, 0.3, 0.4, 0.5),
#   cp = c(-1)
# )

# Valores del experimento 420_01
# PARAM$grid <- list(
#   feature_fraction = c(0.3, 0.5, 0.7),
#   maxdepth = c(6, 10),
#   minsplit = c(500, 200),
#   minbucket_fraction = c(0.2, 0.4),
#   cp = c(-1)
# )

# Valores del experimento 420_02
# PARAM$grid <- list(
#   feature_fraction = c(0.70),
#   maxdepth = c(12, 14, 16),
#   minsplit = c(100, 600, 800),
#   minbucket_fraction = c(0.2, 0.3),
#   cp = c(-1)
# )

# Valores del experimento 420_03
# PARAM$grid <- list(
#   feature_fraction = c(0.20, 0.25, 0.15),
#   maxdepth = c(12, 14, 16, 10),
#   minsplit = c(600, 800, 100, 400),
#   minbucket_fraction = c(0.2, 0.3),
#   cp = c(-1)
# )

# Valores del experimento 420_04
# PARAM$grid <- list(
#   feature_fraction = c(0.10, 0.15, 0.18),    
#   maxdepth = c(14, 16, 18, 20),              
#   minsplit = c(100, 80, 60, 40),             
#   minbucket_fraction = c(0.2),               
#   cp = c(-1)
# )

# Valores del experimento 420_05
PARAM$grid <- list(
  feature_fraction = c(0.15, 0.17, 0.19),    
  maxdepth = c(14, 16, 18, 20),              
  minsplit = c(70, 80, 100, 120),             
  minbucket_fraction = c(0.2),               
  cp = c(-1)
)


In [6]:
# carpeta de trabajo local
# por favor cambiar numero de experimento si se cambia el loop principal
find_project_root <- function() {
  curr <- normalizePath(getwd(), winslash = "/")
  while (curr != "/" && curr != dirname(curr)) {
    if (file.exists(file.path(curr, "Dockerfile")) || dir.exists(file.path(curr, "src"))) {
      return(curr)
    }
    curr <- dirname(curr)
  }
  return(normalizePath(getwd(), winslash = "/"))
}

dir_base <- if (dir.exists("/workspace")) "/workspace" else find_project_root()
experimento <- "exp420_05"
dir_experimento <- file.path(dir_base, "exp", experimento)
dir.create(dir_experimento, showWarnings = FALSE, recursive = TRUE)
setwd(dir_experimento)

In [7]:
# lectura del dataset desde la carpeta local
find_project_root <- function() {
  curr <- normalizePath(getwd(), winslash = "/")
  while (curr != "/" && curr != dirname(curr)) {
    if (file.exists(file.path(curr, "Dockerfile")) || dir.exists(file.path(curr, "src"))) {
      return(curr)
    }
    curr <- dirname(curr)
  }
  return(normalizePath(getwd(), winslash = "/"))
}

dir_base <- if (dir.exists("/workspace")) "/workspace" else find_project_root()
archivo_dataset <- file.path(dir_base, "datasets", "dataset_pequeno.csv")

dataset <- fread(archivo_dataset)

In [8]:
# defino los dataset de entrenamiento y aplicacion
dtrain <- dataset[foto_mes == 202107]
dfuture <- dataset[foto_mes == 202109]

# arreglo clase_ternaria por algun distraido ""
dfuture[, clase_ternaria := NA ]

In [9]:
# Establezco cuales son los campos que puedo usar para la prediccion
# el copy() es por la Lazy Evaluation
campos_buenos <- copy(setdiff(colnames(dtrain), c("clase_ternaria")))

In [10]:
# Bucle anidado para iterar sobre todos los hiperparámetros
iter <- 0

# Tabla para registrar el resumen de todas las corridas
tb_resumen_corridas <- data.table(
  iter = integer(),
  feature_fraction = numeric(),
  maxdepth = integer(),
  minsplit = integer(),
  minbucket_fraction = numeric(),
  minbucket = integer(),
  cp = numeric(),
  num_trees = integer(),
  archivo = character(),
  envios_positivos = integer()
)

for (v_ff in PARAM$grid$feature_fraction) {
  for (v_maxdepth in PARAM$grid$maxdepth) {
    for (v_minsplit in PARAM$grid$minsplit) {
      for (v_minbucket_frac in PARAM$grid$minbucket_fraction) {
        
        # Calcular minbucket como fracción de minsplit (al menos 1)
        v_minbucket <- max(1L, as.integer(round(v_minsplit * v_minbucket_frac)))

        for (v_cp in PARAM$grid$cp) {
          iter <- iter + 1
          
          # Inicializar acumulador de probabilidades para dfuture
          tb_prediccion <- dfuture[, list(numero_de_cliente)]
          tb_prediccion[, prob_acumulada := 0]
          
          # Inicializar semilla para reproducibilidad en cada combinación
          set.seed(PARAM$semilla_primigenia)
          
          param_rpart <- list(
            cp = v_cp,
            maxdepth = v_maxdepth,
            minsplit = v_minsplit,
            minbucket = v_minbucket
          )
          
          cat(sprintf("\n[Iteración %d] ff: %.2f | maxdepth: %d | minsplit: %d | minbucket_frac: %.2f (mb: %d) | cp: %.2f\n",
                      iter, v_ff, v_maxdepth, v_minsplit, v_minbucket_frac, v_minbucket, v_cp))
          
          # Generar los 32 arbolitos del ensemble
          for (arbolito in seq(PARAM$num_trees_max)) {
            qty_campos_a_utilizar <- as.integer(length(campos_buenos) * v_ff)
            
            campos_random <- sample(campos_buenos, qty_campos_a_utilizar)
            campos_random <- paste(campos_random, collapse = " + ")
            formulita <- paste0("clase_ternaria ~ ", campos_random)
            
            modelo <- rpart(formulita,
              data = dtrain,
              xval = 0,
              control = param_rpart
            )
            
            prediccion <- predict(modelo, dfuture, type = "prob")
            tb_prediccion[, prob_acumulada := prob_acumulada + prediccion[, "BAJA+2"]]
          }
          
          # Calcular corte para los 32 árboles (umbral 1/40 acumulado)
          umbral_corte <- (1 / 40) * PARAM$num_trees_max
          tb_prediccion[, Predicted := as.numeric(prob_acumulada > umbral_corte)]
          
          # Nombre descriptivo del archivo con los parámetros utilizados
          archivo_prediccion <- sprintf(
            "KA420_ff_%.2f_md_%d_ms_%d_mb_%d_cp_%.1f_32trees.csv",
            v_ff, v_maxdepth, v_minsplit, v_minbucket, v_cp
          )
          
          # Guardar archivo de predicción
          fwrite(
            tb_prediccion[, list(numero_de_cliente, Predicted)],
            file = archivo_prediccion,
            sep = ","
          )
          
          cant_positivos <- tb_prediccion[, sum(Predicted)]
          cat(sprintf("  -> Guardado: %s (Envíos positivos: %d)\n", archivo_prediccion, cant_positivos))
          
          # Registrar corrida en tabla de resumen
          tb_resumen_corridas <- rbind(
            tb_resumen_corridas,
            list(
              iter = iter,
              feature_fraction = v_ff,
              maxdepth = v_maxdepth,
              minsplit = v_minsplit,
              minbucket_fraction = v_minbucket_frac,
              minbucket = v_minbucket,
              cp = v_cp,
              num_trees = PARAM$num_trees_max,
              archivo = archivo_prediccion,
              envios_positivos = cant_positivos
            )
          )
          
          # Guardar tabla de resumen a disco
          fwrite(tb_resumen_corridas, file = "resumen_corridas.txt", sep = "\t")
        }
      }
    }
  }
}


[Iteración 1] ff: 0.15 | maxdepth: 14 | minsplit: 70 | minbucket_frac: 0.20 (mb: 14) | cp: -1.00
  -> Guardado: KA420_ff_0.15_md_14_ms_70_mb_14_cp_-1.0_32trees.csv (Envíos positivos: 10277)

[Iteración 2] ff: 0.15 | maxdepth: 14 | minsplit: 80 | minbucket_frac: 0.20 (mb: 16) | cp: -1.00
  -> Guardado: KA420_ff_0.15_md_14_ms_80_mb_16_cp_-1.0_32trees.csv (Envíos positivos: 10297)

[Iteración 3] ff: 0.15 | maxdepth: 14 | minsplit: 100 | minbucket_frac: 0.20 (mb: 20) | cp: -1.00
  -> Guardado: KA420_ff_0.15_md_14_ms_100_mb_20_cp_-1.0_32trees.csv (Envíos positivos: 10305)

[Iteración 4] ff: 0.15 | maxdepth: 14 | minsplit: 120 | minbucket_frac: 0.20 (mb: 24) | cp: -1.00
  -> Guardado: KA420_ff_0.15_md_14_ms_120_mb_24_cp_-1.0_32trees.csv (Envíos positivos: 10323)

[Iteración 5] ff: 0.15 | maxdepth: 16 | minsplit: 70 | minbucket_frac: 0.20 (mb: 14) | cp: -1.00
  -> Guardado: KA420_ff_0.15_md_16_ms_70_mb_14_cp_-1.0_32trees.csv (Envíos positivos: 10322)

[Iteración 6] ff: 0.15 | maxdepth: 16 | 

In [11]:
# Mostrar resumen de las corridas realizadas
tb_resumen_corridas

iter,feature_fraction,maxdepth,minsplit,minbucket_fraction,minbucket,cp,num_trees,archivo,envios_positivos
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<chr>,<dbl>
1,0.15,14,70,0.2,14,-1,32,KA420_ff_0.15_md_14_ms_70_mb_14_cp_-1.0_32trees.csv,10277
2,0.15,14,80,0.2,16,-1,32,KA420_ff_0.15_md_14_ms_80_mb_16_cp_-1.0_32trees.csv,10297
3,0.15,14,100,0.2,20,-1,32,KA420_ff_0.15_md_14_ms_100_mb_20_cp_-1.0_32trees.csv,10305
4,0.15,14,120,0.2,24,-1,32,KA420_ff_0.15_md_14_ms_120_mb_24_cp_-1.0_32trees.csv,10323
5,0.15,16,70,0.2,14,-1,32,KA420_ff_0.15_md_16_ms_70_mb_14_cp_-1.0_32trees.csv,10322
6,0.15,16,80,0.2,16,-1,32,KA420_ff_0.15_md_16_ms_80_mb_16_cp_-1.0_32trees.csv,10343
7,0.15,16,100,0.2,20,-1,32,KA420_ff_0.15_md_16_ms_100_mb_20_cp_-1.0_32trees.csv,10301
8,0.15,16,120,0.2,24,-1,32,KA420_ff_0.15_md_16_ms_120_mb_24_cp_-1.0_32trees.csv,10345
9,0.15,18,70,0.2,14,-1,32,KA420_ff_0.15_md_18_ms_70_mb_14_cp_-1.0_32trees.csv,10363


In [12]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Mon Aug 24 23:02:36 2026"



---

